In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [ ]:
from google.colab import files
upload = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
df = pd.read_csv('silver_daily_ohlcv_2000_2025.csv')
print(df.columns.tolist())
print(df.head(2))

In [ ]:
print(df.columns.tolist())

In [ ]:
covid = df[(df['Date'] >= '2020-01-01') & (df['Date'] <= '2020-12-31')].copy()
print(f"COVID period rows: {len(covid)}")
print(f"Price range: ${covid['Low'].min():.2f} - ${covid['High'].max():.2f}")
print(f"Start price: ${covid['Close'].iloc[0]:.2f}")
print(f"End price: ${covid['Close'].iloc[-1]:.2f}")

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
covid = df[(df['Date'] >= '2020-01-01') & (df['Date'] <= '2020-12-31')].copy()
covid = covid.reset_index(drop=True)
print(f"COVID period rows: {len(covid)}")
print(f"Price range: ${covid['Low'].min():.2f} - ${covid['High'].max():.2f}")
print(f"Start price: ${covid['Close'].iloc[0]:.2f}")
print(f"End price: ${covid['Close'].iloc[-1]:.2f}")

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Train on pre-COVID data, test on COVID period
train = df[df['Date'] < '2020-01-01']['Close']
test = covid['Close']

# ARIMA model
model = ARIMA(train, order=(5,1,0))
fitted = model.fit()

# Forecast for COVID period
forecast = fitted.forecast(steps=len(test))

# Metrics
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
print(f"ARIMA MAE: ${mae:.2f}")
print(f"ARIMA RMSE: ${rmse:.2f}")
print(f"Forecast range: ${forecast.min():.2f} - ${forecast.max():.2f}")
print(f"Actual range: ${test.min():.2f} - ${test.max():.2f}")


In [ ]:
# Simple Moving Average model
sma_20 = train.rolling(20).mean().iloc[-1]  # Last SMA value
sma_forecast = np.full(len(test), sma_20)

mae_sma = mean_absolute_error(test, sma_forecast)
rmse_sma = np.sqrt(mean_squared_error(test, sma_forecast))
print(f"SMA Model MAE: ${mae_sma:.2f}")
print(f"SMA Model RMSE: ${rmse_sma:.2f}")
print(f"SMA Forecast value: ${sma_20:.2f}")

In [ ]:
print("=== CASE STUDY C3: COVID-19 MODEL PERFORMANCE ===")
print(f"Study period: Jan 2020 - Dec 2020")
print(f"Silver price at start: $18.77")
print(f"Silver crash low: $11.70 (March 18, 2020)")
print(f"Silver recovery high: $29.64 (August 2020)")
print(f"Total recovery: +151% from low")
print(f"\nARIMA: MAE=${4.49:.2f}, RMSE=${5.15:.2f} - FAILED (flat forecast)")
print(f"SMA-20: MAE=${4.54:.2f}, RMSE=${5.25:.2f} - FAILED (flat forecast)")
print(f"\nKey insight: Both models predicted ~$18.50 throughout")
print(f"Actual range was $11.70-$29.64 - a $17.94 spread!")

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(covid['Date'], covid['Close'], label='Actual Price', color='#1F4E79', linewidth=2)
plt.axhline(y=18.63, color='#CC0000', linestyle='--', label=f'ARIMA Forecast ($18.63)')
plt.axhline(y=18.39, color='#228B22', linestyle='--', label=f'SMA Forecast ($18.39)')
plt.axvline(x=pd.Timestamp('2020-03-18'), color='orange', linestyle=':', label='Crash Low ($11.70)')
plt.axvline(x=pd.Timestamp('2020-08-07'), color='purple', linestyle=':', label='Recovery Peak ($29.64)')
plt.title('COVID-19 Silver Crash 2020 — Model Failure Analysis')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig('covid_model_analysis.png', dpi=150)
plt.show()
print("Chart saved!")

In [ ]:
# Check train/test split - no leakage
train = df[df['Date'] < '2020-01-01'].copy()
test = df[df['Date'] >= '2020-01-01'].copy()

print(f"Train size: {len(train)} rows")
print(f"Test size: {len(test)} rows")
print(f"Train date range: {train['Date'].min()} to {train['Date'].max()}")
print(f"Test date range: {test['Date'].min()} to {test['Date'].max()}")
print(f"Overlap check: {len(train[train['Date'].isin(test['Date'])])} overlapping dates")